### Import library and Dataset

In [9]:
# import library
import numpy as np
import pandas as pd

# import dataset
df = pd.read_csv("car_fuel_efficiency_2026.csv")

### Q1. Check column with missing value

In [10]:
# Q1. check column with missing value

# this is the columns name in the quiz answer options
selected_columns = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']

# columns with missing value
column_with_missing_value = df.columns[df.isnull().any()].tolist()
print(f'columns with missing value: {column_with_missing_value}')

missing_value = []

for i in range(len(column_with_missing_value)):
    if column_with_missing_value[i] in selected_columns:
        missing_value.append(column_with_missing_value[i])

if len(missing_value) >= 0:
    print(f'Columns that have missing values that match the quiz answer options is: {missing_value}')
else:
    print(f'The dataset does not have columns with missing values')

columns with missing value: ['horsepower', 'acceleration']
Columns that have missing values that match the quiz answer options is: ['horsepower']


### Q2. Median for column `horsepower`

In [11]:
# Q2. median for column horsepower
median_horsepower = df['horsepower'].median()
print(f'Median (50% percentile) for horsepower: {median_horsepower}')

Median (50% percentile) for horsepower: 254.0


In [12]:
# Shuffle the filtered dataset and create the split exactly as in the lecture:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

### Q3. Best Option for filling NAs

In [13]:
# Q3. best option for filling NAs
def rmse(y, y_pred):
    error = y_pred - y
    mse = (error ** 2).mean()
    return np.sqrt(mse)

def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    return w[0], w[1:]

features = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']

# Fill with 0
X_train_0 = df_train[features].fillna(0).values
w0_0, w_0 = train_linear_regression(X_train_0, df_train['fuel_efficiency_mpg'].values)
X_val_0 = df_val[features].fillna(0).values
y_pred_0 = w0_0 + X_val_0.dot(w_0)
rmse_0 = rmse(df_val['fuel_efficiency_mpg'].values, y_pred_0)

# Fill with mean
mean_hp = df_train['horsepower'].mean()
X_train_m = df_train[features].fillna(mean_hp).values
w0_m, w_m = train_linear_regression(X_train_m, df_train['fuel_efficiency_mpg'].values)
X_val_m = df_val[features].fillna(mean_hp).values
y_pred_m = w0_m + X_val_m.dot(w_m)
rmse_m = rmse(df_val['fuel_efficiency_mpg'].values, y_pred_m)

print(f'RMSE with 0: {round(rmse_0, 3)}')
print(f'RMSE with mean: {round(rmse_m, 3)}')
print('The option that yields a better RMSE value: "With Mean".')

RMSE with 0: 2.205
RMSE with mean: 2.202
The option that yields a better RMSE value: "With Mean".


### Q4. Best regularization parameter r

In [14]:
def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    return w[0], w[1:]

r_values = [0, 0.01, 0.1, 1, 5, 10, 100]
for r in r_values:
    w0, w = train_linear_regression_reg(X_train_0, df_train['fuel_efficiency_mpg'].values, r=r)
    y_pred = w0 + X_val_0.dot(w)
    score = rmse(df_val['fuel_efficiency_mpg'].values, y_pred)
    print(f'{r}: {round(score, 4)}')
print('The r-parameter option that yields the best RMSE: 0')

0: 2.2053
0.01: 2.2058
0.1: 2.2241
1: 2.3492
5: 2.4094
10: 2.4195
100: 2.4292
The r-parameter option that yields the best RMSE: 0


### Q5. STD of RMSE scores for different seeds

In [15]:
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
rmses = []
for s in seeds:
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test
    
    np.random.seed(s)
    idx = np.arange(n)
    np.random.shuffle(idx)
    
    df_train_s = df.iloc[idx[:n_train]]
    df_val_s = df.iloc[idx[n_train:n_train + n_val]]
    df_test_s = df.iloc[idx[n_train + n_val:]]
    
    X_train_s = df_train_s[features].fillna(0).values
    y_train_s = df_train_s['fuel_efficiency_mpg'].values
    w0_s, w_s = train_linear_regression(X_train_s, y_train_s)
    
    X_val_s = df_val_s[features].fillna(0).values
    y_val_s = df_val_s['fuel_efficiency_mpg'].values
    y_pred_s = w0_s + X_val_s.dot(w_s)
    
    rmses.append(rmse(y_val_s, y_pred_s))

print(f'Std: {round(np.std(rmses), 3)}')
print('The STD value: 0.029')

Std: 0.029
The STD value: 0.029


### Q6. RMSE on test dataset with seed 9 and r=0.001

In [16]:
np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)

df_train_9 = df.iloc[idx[:n_train]]
df_val_9 = df.iloc[idx[n_train:n_train + n_val]]
df_test_9 = df.iloc[idx[n_train + n_val:]]

df_full_train = pd.concat([df_train_9, df_val_9])
X_full_train = df_full_train[features].fillna(0).values
y_full_train = df_full_train['fuel_efficiency_mpg'].values

w0_full, w_full = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)

X_test_9 = df_test_9[features].fillna(0).values
y_test_9 = df_test_9['fuel_efficiency_mpg'].values
y_pred_test_9 = w0_full + X_test_9.dot(w_full)

print(f'RMSE on test: {round(rmse(y_test_9, y_pred_test_9), 3)}')
print('RMSE of the test dataset: 2.236')

RMSE on test: 2.236
RMSE of the test dataset: 2.236
